<a href="https://colab.research.google.com/github/anathayna/tcc/blob/main/tcc_fim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color="orange">**TCC: Identificação do discurso de ódio em memes**</font>

O Google Colab é uma plataforma baseada em Jupyter Notebook que oferece um ambiente de execução em nuvem com recursos gratuitos de GPU e TPU, amplamente utilizado em projetos de aprendizado de máquina e ciência de dados.

Este notebook documenta a etapa final do desenvolvimento do trabalho de conclusão de curso, cujo objetivo é identificar e classificar discursos de ódio em memes por meio de técnicas de aprendizado de máquina multimodal.

# <font color="orange">**Sumário**</font>

1.   Instalando bibliotecas e dependências
2.   Preparando o banco de dados
4.   Treinamento
5.   Classificação
6.   Avaliação
7.   Conclusão

## <font color="orange">1. Instalando bibliotecas e dependências</font>

In [1]:
!pip install ftfy regex tqdm

In [2]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-hrwos4xu
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-hrwos4xu
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done


In [3]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [4]:
!conda create -n RGCL python=3.10 -y
!conda activate RGCL

Channels:
 - conda-forge
Platform: linux-64
Solving environment: / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/RGCL

  added / updated specs:
    - python=3.10


The following NEW packages will be INSTALLED:

  _openmp_mutex      conda-forge/linux-64::_openmp_mutex-4.5-20_gnu 
  bzip2              conda-forge/linux-64::bzip2-1.0.8-hda65f42_9 
  ca-certificates    conda-forge/noarch::ca-certificates-2026.4.22-hbd8a1cb_0 
  icu                conda-forge/linux-64::icu-78.3-h33c6efd_0 
  ld_impl_linux-64   conda-forge/linux-64::ld_impl_linux-64-2.45.1-default_hbd61a6d_102 
  libexpat           conda-forge/linux-64::libexpat-2.7.5-hecca717_0 
  libffi             conda-forge/linux-64::libffi-3.5.2-h3435931_0 
  libgcc             conda-forge/linux-64::libgcc-15.2.0-he0feb66_

In [5]:
!conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia -y

Channels:
 - pytorch
 - nvidia
 - conda-forge
Platform: linux-64
Solving environment: \ | / - \ done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.



In [6]:
!conda install -c pytorch -c nvidia faiss-gpu=1.7.4 mkl=2021 blas=1.0=mkl -y

Channels:
 - pytorch
 - nvidia
 - conda-forge
Platform: linux-64
Solving environment: | / - \ done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.



In [7]:
!rm -rf RGCL
!git clone https://github.com/JingbiaoMei/RGCL.git
%cd RGCL
!pip install -r requirement.txt

Cloning into 'RGCL'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 225 (delta 27), reused 28 (delta 24), pack-reused 193 (from 1)
Receiving objects: 100% (225/225), 111.04 KiB | 4.27 MiB/s, done.
Resolving deltas: 100% (122/122), done.
/content/RGCL
  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


In [8]:
!pip install jsonlines

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import torch
import clip
import os

In [10]:
!pip install open-clip-torch

## <font color="orange">2. Preparando o banco de dados</font>

O banco de dados utilizado para a identificação do discurso de ódio em memes é o **Hateful Memes** disponibilizado pela Meta no *Challenge Hateful Memes*, um banco de dados com mais de **10 mil** imagens de memes em inglês, que contêm conteúdo ofensivo relacionado a gênero, raça, religião, orientação sexual, classe social e outros tópicos.

E o conjunto de dados é composto pelas seguintes porcentagens:

- **40%** de memes de ódio multimodal (multimodal hate): memes em que tanto o texto quanto a imagem contribuem para a mensagem de ódio.

- **10%** de memes de ódio unimodal (unimodal hate): memes em que apenas uma das modalidades (texto ou imagem) é suficiente para transmitir o discurso de ódio.

- **20%** de memes com confusão de texto benigna (benign text confounder): memes em que o texto foi alterado para remover o discurso de ódio, mas a imagem ainda pode sugerir um significado ofensivo.

- **20%** de memes com confusão de imagem benigna (benign image confounder): memes em que a imagem foi alterada para remover o discurso de ódio, mas o texto ainda pode sugerir um significado ofensivo.

- **10%** de memes não odiosos aleatórios (random non-hateful): memes que não contêm discurso de ódio, escolhidos aleatoriamente.


Um **confundidor benigno** é, basicamente, uma alteração mínima feita em um meme (seja mudando o texto ou a imagem) que faz com que a classificação dele mude de “odioso” para “não odioso”. A alteração é a menor possível, justamente para garantir que a diferença entre o meme odioso e o benigno seja bem sutil e dependa de uma análise genuína das duas modalidades (texto+imagem).


![](https://drivendata-public-assets.s3.amazonaws.com/memes-overview.png)


**Figura 1:** Exemplo de meme utilizado na competição  
Fonte: DRIVENDATA (2020)

Na figura 1, é possível visualizar que o confundidor benigno de texto altera o texto e mantém a imagem, enquanto o confundidor benigno de imagem altera a imagem e mantém o texto.

Baixe o banco de dados do **Hateful Memes** pela plataforma *Kaggle* no seguinte endereço: https://www.kaggle.com/datasets/williamberrios/hateful-memes

Originalmente, esse conjunto de dados estava disponível no site oficial do desafio (https://hatefulmemeschallenge.com/#download). Entretanto, no momento da elaboração desta pesquisa, o domínio encontrava-se indisponível.

In [11]:
#@markdown Defina o caminho para o arquivo **.zip** do banco de dados do *Hateful Memes*.
#@markdown **exemplo:** `"/content/drive/MyDrive/hateful_memes.zip"`

PATH_TO_ZIP_FILE = '/content/drive/MyDrive/hateful_memes.zip' #@param {type:"string"}

#@markdown Defina o diretório base para extrair o banco de dados.
#@markdown **exemplo:** `"/content"`

HOME = '/content' #@param {type:"string"}

Após realizar o download do banco de dados e armazená-lo no Google Drive, é necessário integrá-lo ao ambiente do Google Colab para extração das características. Para isso, utilizamos a biblioteca google.colab.drive, que permite montar o Google Drive como um sistema de arquivos virtual no ambiente de execução.

Ao executar o código, o Colab solicitará uma autorização para acessar sua conta do Google Drive. Uma vez concedida a permissão, o Drive será vinculado ao diretório `/content/drive/`, possibilitando a leitura dos arquivos armazenados. Essa abordagem garante acesso contínuo aos dados durante o processamento, sem necessidade de uploads manuais.

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Agora, preparamos o ambiente para incorporar a pasta "model", que armazenará o banco de dados e outros arquivos essenciais, garantindo que o projeto possa acessá-los corretamente.

In [13]:
import os
os.chdir(HOME)
os.getcwd()
os.environ['PYTHONPATH'] += ":/content/model/"

Em seguida, o código realiza a extração automática do banco de dados Hateful Memes (armazenado no formato `.zip`) para o diretório `/content/model/`, disponibilizando os arquivos para as próximas etapas de processamento e análise no notebook.

In [14]:
import zipfile
zip_path = PATH_TO_ZIP_FILE
extract_path = '/content/model/'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [15]:
img_dir = '/content/model/hateful_memes/img'

image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp']

image_count = 0
for filename in os.listdir(img_dir):
    if any(filename.lower().endswith(ext) for ext in image_extensions):
        image_count += 1

print(f"total de imagens: {image_count}")

total de imagens: 12140


In [16]:
import os
import shutil
from tqdm.auto import tqdm

rgcl_data_dir = '/content/RGCL/data/image/FB/All/'

img_dir = '/content/model/hateful_memes/img'

os.makedirs(rgcl_data_dir, exist_ok=True)

print("Copiando imagens para a estrutura do RGCL...")
images = [f for f in os.listdir(img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
for img_file in tqdm(images):
    src = os.path.join(img_dir, img_file)
    dst = os.path.join(rgcl_data_dir, img_file)
    shutil.copy(src, dst)

print(f"Total de {len(images)} imagens copiadas para {rgcl_data_dir}")

Copiando imagens para a estrutura do RGCL...


  0%|          | 0/12140 [00:00<?, ?it/s]

Total de 12140 imagens copiadas para /content/RGCL/data/image/FB/All/


In [17]:
import os
import shutil
import glob

rgcl_gt_dir = '/content/RGCL/data/gt/FB/'
os.makedirs(rgcl_gt_dir, exist_ok=True)

base_path = '/content/model/hateful_memes/'

jsonl_files = glob.glob(os.path.join(base_path, '*.jsonl'))

print(f"Copiando arquivos de anotação para {rgcl_gt_dir}...")
for file_path in jsonl_files:
    file_name = os.path.basename(file_path)
    dst_path = os.path.join(rgcl_gt_dir, file_name)
    shutil.copy(file_path, dst_path)
    print(f"Copiado: {file_name}")

print("Processo concluído!")

Copiando arquivos de anotação para /content/RGCL/data/gt/FB/...
Copiado: train.jsonl
Copiado: dev_unseen.jsonl
Copiado: test_unseen.jsonl
Copiado: dev_seen.jsonl
Copiado: test_seen.jsonl
Processo concluído!


In [24]:
%cd /content/RGCL
!python3 src/utils/generate_CLIP_embedding_HF.py --dataset "FB"

/content/RGCL
Namespace(EXP_FOLDER='./data/CLIP_Embedding', model='openai/clip-vit-large-patch14-336', image_size=336, dataset='FB', batch_size=32, all=False)
config.json: 4.76kB [00:00, 13.8MB/s]
pytorch_model.bin: 100% 1.71G/1.71G [00:06<00:00, 251MB/s]
Loading weights: 100% 391/391 [00:00<00:00, 20724.78it/s]
[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14-336
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.

In [19]:
!pip install --upgrade torch torchvision torchaudio

In [20]:
%cd /content/RGCL
!python3 src/utils/generate_ALIGN_embedding_HF.py --dataset "FB"

/content/RGCL
Namespace(EXP_FOLDER='./data/CLIP_Embedding', model='kakaobrain/align-base', image_size=346, dataset='FB', batch_size=32, all=False, trunc=True, token_paral=False)
Loading weights: 100% 1394/1394 [00:00<00:00, 25467.86it/s]
tokenizer_config.json: 100% 399/399 [00:00<00:00, 2.77MB/s]
vocab.txt: 232kB [00:00, 15.0MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 909kB/s]
preprocessor_config.json: 100% 508/508 [00:00<00:00, 3.59MB/s]
/usr/local/lib/python3.11/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 24 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.11/site-packages/torch/utils/data/datalo

In [23]:
%cd /content/RGCL
!MPLBACKEND=agg bash scripts/experiments.sh

/content/RGCL
Namespace(path='./data/', output_path='./logging/Retrieval/FB/RAC/RAC_lr0.0001_Bz64_Ep30_cosSim_triplet_drop[0.2, 0.4, 0.1]_topK20__PseudoGold_positive_1_hard_negative_1_seed0_hybrid_loss/', model='openai_clip-vit-large-patch14-336_HF', dataset='FB', similarity_threshold=-1.0, fusion_mode='align', topk=20, majority_voting='arithmetic', metric='cos', loss='triplet', triplet_margin=0.1, norm_feats_loss=False, l2_sqrt=False, hybrid_loss=True, ce_weight=0.5, pos_weight_value=None, num_layers=3, proj_dim=1024, map_dim=1024, dropout=[0.2, 0.4, 0.1], batch_norm=False, last_layer='none', epochs=30, batch_size=64, lr=0.0001, weight_decay=0.0001, lr_scheduler=False, num_workers=24, grad_clip=0.1, no_pseudo_gold_positives=1, in_batch_loss=True, hard_negatives_loss=True, no_hard_negatives=1, no_hard_positives=0, hard_negatives_multiple=12, Faiss_GPU=True, reindex_every_step=False, sparse_dictionary=None, use_attribute=True, sparse_topk=None, eval_retrieval=True, log_interval=10, fina

## <font color="orange">4. Treinamento</font>

## <font color="orange">5. Classificação</font>

## <font color="orange">6. Avaliação</font>

Nesta seção, são apresentados os resultados da avaliação do modelo. Para interpretar o desempenho obtido, são empregadas métricas amplamente utilizadas em tarefas de classificação supervisionada, como acurácia, precisão, revocação e F1-score, descritas a seguir.

**Acurácia**:
Mede a proporção total de acertos, ou seja, quantas previsões o modelo acertou em relação ao total de exemplos avaliados.

**Precisão** (Precision):
Indica que entre as amostras que o modelo previu como “ódio”, quantas realmente pertencem a essa classe.

Alta precisão significa poucos falsos positivos (erros ao classificar “não ódio” como “ódio”).

**Revocação** (Recall):
Mede entre os memes que realmente são “ódio”, quantos o modelo conseguiu identificar corretamente.

Alta revocação significa poucos falsos negativos (casos de ódio que o modelo deixou passar).

**F1-score**:
É a média harmônica entre precisão e revocação, equilibrando as duas métricas. É útil quando há diferença de desempenho entre as classes.

**Support**:
Mostra a quantidade de amostras de cada classe utilizadas no cálculo das métricas.

A matriz de confusão permite visualizar o desempenho do modelo em termos de acertos e erros para cada classe.

Ela mostra quantas amostras de cada classe foram corretamente classificadas e quantas foram confundidas com a outra categoria, facilitando a identificação dos tipos de erro mais comuns.

- **Verdadeiros Positivos (True Positives - TP):** Memes que são de ódio (rótulo verdadeiro = 1) e foram corretamente classificados como ódio (predição = 1).
- **Verdadeiros Negativos (True Negatives - TN):** Memes que não são de ódio (rótulo verdadeiro = 0) e foram corretamente classificados como não ódio (predição = 0).
- **Falsos Positivos (False Positives - FP):** Memes que não são de ódio (rótulo verdadeiro = 0) mas foram classificados incorretamente como ódio (predição = 1).
- **Falsos Negativos (False Negatives - FN):** Memes que são de ódio (rótulo verdadeiro = 1) mas foram classificados incorretamente como não ódio (predição = 0).

A matriz de confusão indica...

A lógica *fuzzy* permite que, em vez de um rótulo binário (0 ou 1), atribuímos graus de pertinência a diferentes categorias ou conceitos. As próprias probabilidades (`probs`) que o modelo CLIP gera para "isso é ódio" e "isso não é ódio" já podem ser vistas como graus de pertinência a esses conjuntos *fuzzy*.

A seguir, são mostrados exemplos de acertos e erros do modelo, permitindo uma análise visual do comportamento do CLIP. Essa etapa ajuda a entender em quais situações o modelo acerta e onde tende a confundir as classes.

## <font color="orange">7. Conclusão</font>

## <font color="orange">Teste</font>

In [ ]:
!pip install --upgrade google-cloud-vision -q

In [ ]:
!pip install deep-translator -q

In [ ]:
import gradio as gr
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.feature_extraction import create_feature_extractor
from sklearn.preprocessing import normalize
import clip
from deep_translator import GoogleTranslator
import random
import os
import io
from google.cloud import vision
from google.colab import userdata
import tempfile

gr.close_all()

try:
    key_json = userdata.get('GOOGLE_CLOUD_KEY')
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        f.write(key_json)
        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = f.name
except Exception as e:
    print("Aviso: Chave GOOGLE_CLOUD_KEY não encontrada nos Secrets do Colab.", e)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

try:
    vision_client = vision.ImageAnnotatorClient()
except Exception as e:
    print("Aviso: Falha ao inicializar o Vision API. Verifique suas credenciais.", e)
    vision_client = None

resnet_model = resnet18(weights=ResNet18_Weights.DEFAULT).eval().to(device)
feat_extractor_single = create_feature_extractor(resnet_model, return_nodes={'avgpool':'features'})
resnet_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def classify_meme(image):
    if image is None:
        return "Nenhuma imagem fornecida.", "", "Por favor, faça o upload de uma imagem."

    text = ""
    if vision_client:
        try:
            byte_array = io.BytesIO()
            image.save(byte_array, format='PNG')
            content = byte_array.getvalue()
            vision_image = vision.Image(content=content)

            response = vision_client.text_detection(image=vision_image)
            texts = response.text_annotations

            if response.error.message:
                print(f"Erro na API Vision (Resposta): {response.error.message}")
            elif texts:
                text = texts[0].description.replace('\n', ' ').strip()
        except Exception as e:
            print(f"Erro na API Vision (Exceção): {e}")
            text = ""

    if not text:
        text = "no text"
        texto_traduzido = "sem texto"
    else:
        try:
            texto_traduzido = GoogleTranslator(source='auto', target='pt').translate(text)
        except Exception as e:
            texto_traduzido = "erro na tradução"

    img_res_tensor = resnet_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        img_feats = feat_extractor_single(img_res_tensor)['features'].squeeze()
        if img_feats.dim() == 0:
            img_feats = img_feats.unsqueeze(0)
        img_feats = img_feats / (img_feats.norm(dim=0, keepdim=True) + 1e-10)
        img_feats_np = img_feats.cpu().numpy().reshape(1, -1)

    _, img_idx = nbrs_img.kneighbors(img_feats_np)
    c_img = ref_cluster_img_ids[img_idx[0][0]]

    txt_feats = text_encoder.encode([text], convert_to_numpy=True)
    txt_feats = normalize(txt_feats, axis=1)
    _, text_idx = nbrs_text.kneighbors(txt_feats)
    c_text = ref_cluster_text_ids[text_idx[0][0]]

    image_clip = clip_preprocess(image).unsqueeze(0).to(device)
    text_clip = clip.tokenize([text], truncate=True).to(device)
    c_text_tensor = torch.tensor([c_text], dtype=torch.long).to(device)
    c_img_tensor = torch.tensor([c_img], dtype=torch.long).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(image_clip, text_clip, c_text_tensor, c_img_tensor)
        prob_odio = torch.sigmoid(logits).item()

    prob_nao_odio = 1.0 - prob_odio

    fuzzy_result = unified_fuzzy_classifier.classify(prob_odio, prob_nao_odio)
    categoria = fuzzy_result['categoria_fuzzy']

    resultado = (
        f"### 🚨 Análise Concluída\n\n"
        f"- **Probabilidade de Ódio:** {prob_odio*100:.2f}%\n"
        f"- **Probabilidade de Não Ódio:** {prob_nao_odio*100:.2f}%\n\n"
        f"**📌 Classificação Final (Fuzzy):** {categoria}"
    )
    return text, texto_traduzido, resultado

def load_random_meme():
    row = df_results.sample(1).iloc[0]
    img_path = os.path.join(base_path, row['img'])
    img = Image.open(img_path).convert('RGB')

    texto = row['text']
    try:
        texto_traduzido = GoogleTranslator(source='auto', target='pt').translate(texto)
    except Exception:
        texto_traduzido = "erro na tradução"

    md_traducao = f"**Tradução do Meme:** {texto_traduzido}"

    return img, row.to_dict(), "", None, md_traducao

def check_guess(guess, row_data):
    if not row_data:
        return "⚠️ Por favor, clique em 'Carregar Meme Aleatório' primeiro."
    if not guess:
        return "⚠️ Por favor, selecione um palpite antes de verificar."

    real_label = "Ódio" if row_data['true_label'] == 1 else "Não Ódio"
    modelo_fuzzy = row_data['categoria_fuzzy']
    prob_odio = row_data['prob_ódio'] * 100
    texto = row_data['text']

    try:
        texto_traduzido = GoogleTranslator(source='auto', target='pt').translate(texto)
    except Exception as e:
        texto_traduzido = "erro na tradução"

    md = f"### 🎯 Resultado da Verificação\n\n"
    md += f"- **Seu Palpite:** {guess}\n"
    md += f"- **Rótulo Real (Ground Truth):** {real_label}\n"
    md += f"- **O que o Modelo Previu:** {modelo_fuzzy} (Probabilidade de Ódio: {prob_odio:.2f}%)\n\n"
    return md

with gr.Blocks(title="TCC: Discurso de Ódio em Memes") as demo:
    gr.Markdown("# TCC: Identificador de Discurso de Ódio em Memes")

    with gr.Tab("Upload de Meme"):
        gr.Markdown("Faça o upload de um meme. O sistema usará IA para extrair o texto automaticamente, traduzi-lo e classificar se contém discurso de ódio usando lógica Fuzzy.")
        with gr.Row():
            with gr.Column():
                img_input = gr.Image(type="pil", label="Faça o Upload do Meme")
                btn_classify = gr.Button("Classificar Meme", variant="primary")
            with gr.Column():
                txt_en = gr.Textbox(label="Texto Extraído (Inglês)")
                txt_pt = gr.Textbox(label="Texto Traduzido (Português)")
                md_result = gr.Markdown(label="Resultado da Análise")

        btn_classify.click(classify_meme, inputs=[img_input], outputs=[txt_en, txt_pt, md_result])

    with gr.Tab("Teste o Modelo (Aleatório)"):
        gr.Markdown("Sorteie um meme do banco de dados, dê o seu palpite e veja se você concorda com o Modelo e com a classificação real!")
        state_meme = gr.State()
        with gr.Row():
            with gr.Column():
                img_random = gr.Image(type="pil", label="Meme Aleatório", interactive=False)
                md_translation = gr.Markdown()
                btn_new_meme = gr.Button("Carregar Novo Meme Aleatório")
            with gr.Column():
                radio_guess = gr.Radio(
                    choices=["Nenhum Ódio", "Pouco Ódio", "Ambiguidade/Moderado", "Ódio", "Alto Grau de Ódio"],
                    label="O que você acha que é?"
                )
                btn_verify = gr.Button("Verificar Resultado", variant="primary")
                md_verify_result = gr.Markdown()

        btn_new_meme.click(load_random_meme, inputs=[], outputs=[img_random, state_meme, md_verify_result, radio_guess, md_translation])
        btn_verify.click(check_guess, inputs=[radio_guess, state_meme], outputs=[md_verify_result])

demo.launch(debug=True, share=True)
